| Dataset | Peran | Output |
|---------|-------|--------|
| Obesity Dataset | ML Training | `obesity_encoded.csv` + artifacts `.pkl` |
| Food & Nutrition | Recommendation DB | `food_clean.csv` |
| Gym Exercise | Recommendation DB | `gym_clean.csv` |

> Hanya Obesity Dataset yang di-encode & di-scale untuk ML.  
> Food & Gym cukup dibersihkan dan distandarisasi sebagai lookup database.

In [1]:
import pandas as pd
import numpy as np
import pickle
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)
print('Directories ready.')

Directories ready.


---
## BAGIAN 1 — Obesity Dataset
### Tujuan: Siapkan data untuk ML Classification

**Langkah:**
1. Drop 24 duplikat
2. Feature engineering: BMI, BMR, DailyCalorieTarget
3. Label encode semua kolom object
4. StandardScaler pada fitur numerik
5. Simpan 2 versi: readable + ML-ready

In [2]:
# ── Load ──────────────────────────────────────────────────────────
obesity = pd.read_csv('..\\data\\raw\\ObesityDataSet_raw_and_data_sinthetic.csv')
print(f'Original shape: {obesity.shape}')  # (2111, 17)

# ── 1. Drop duplicates ────────────────────────────────────────────
obesity.drop_duplicates(inplace=True)
obesity.reset_index(drop=True, inplace=True)
print(f'After drop_duplicates: {obesity.shape}')  # expect (2087, 17)

# ── 2. Feature Engineering ────────────────────────────────────────

# BMI
obesity['BMI'] = (obesity['Weight'] / (obesity['Height'] ** 2)).round(2)

# BMR — Harris-Benedict
def calc_bmr(row):
    h_cm = row['Height'] * 100
    w    = row['Weight']
    age  = row['Age']
    if str(row['Gender']).lower() == 'female':
        return round(447.593 + (9.247 * w) + (3.098 * h_cm) - (4.330 * age), 2)
    else:
        return round(88.362 + (13.397 * w) + (4.799 * h_cm) - (5.677 * age), 2)

obesity['BMR'] = obesity.apply(calc_bmr, axis=1)

# DailyCalorieTarget — FAF: 0=sedentary,1=light,2=moderate,3=active
activity_multiplier = {0: 1.2, 1: 1.375, 2: 1.55, 3: 1.725}
obesity['ActivityLevel'] = obesity['FAF'].apply(lambda x: min(int(round(x)), 3))
obesity['DailyCalorieTarget'] = (
    obesity['BMR'] * obesity['ActivityLevel'].map(activity_multiplier)
).round(0).astype(int)

print('New engineered features: BMI, BMR, ActivityLevel, DailyCalorieTarget')
obesity[['Gender','Age','Height','Weight','BMI','BMR','ActivityLevel','DailyCalorieTarget','NObeyesdad']].head()

Original shape: (2111, 17)
After drop_duplicates: (2087, 17)
New engineered features: BMI, BMR, ActivityLevel, DailyCalorieTarget


,Gender,Age,Height,Weight,BMI,BMR,ActivityLevel,DailyCalorieTarget,NObeyesdad
0,Female,21.0,1.62,64.0,24.39,1450.35,0,1740,Normal_Weight
1,Female,21.0,1.52,56.0,24.24,1345.39,3,2321,Normal_Weight
2,Male,23.0,1.80,77.0,23.77,1853.18,2,2872,Normal_Weight
3,Male,27.0,1.80,87.0,26.85,1964.44,2,3045,Overweight_Level_I
4,Male,22.0,1.78,89.8,28.34,2020.74,0,2425,Overweight_Level_II


In [3]:
# ── Save READABLE version (sebelum encoding) ──────────────────────
# Versi ini digunakan backend untuk menghitung rekomendasi kalori
obesity.to_csv('..\\data\\raw\\ObesityDataSet_raw_and_data_sinthetic_clean.csv', index=False)
print('Saved: obesity_clean.csv  (human-readable, includes engineered features)')

Saved: obesity_clean.csv  (human-readable, includes engineered features)


In [4]:
# ── 3. Label Encoding ─────────────────────────────────────────────
# Kolom object dari EDA:
# Gender, family_history_with_overweight, FAVC, CAEC, SMOKE, SCC, CALC, MTRANS, NObeyesdad

obesity_enc = obesity.copy()
le_dict     = {}

cat_cols = obesity_enc.select_dtypes(include='object').columns.tolist()
print('Encoding columns:', cat_cols)

for col in cat_cols:
    le = LabelEncoder()
    obesity_enc[col] = le.fit_transform(obesity_enc[col].astype(str))
    le_dict[col]     = le
    print(f'  {col}: {dict(enumerate(le.classes_))}')

# Simpan LabelEncoders — dibutuhkan saat inference
with open('../models/label_encoders.pkl', 'wb') as f:
    pickle.dump(le_dict, f)
print('\nSaved: models/label_encoders.pkl')

Encoding columns: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS', 'NObeyesdad']
  Gender: {0: 'Female', 1: 'Male'}
  family_history_with_overweight: {0: 'no', 1: 'yes'}
  FAVC: {0: 'no', 1: 'yes'}
  CAEC: {0: 'Always', 1: 'Frequently', 2: 'Sometimes', 3: 'no'}
  SMOKE: {0: 'no', 1: 'yes'}
  SCC: {0: 'no', 1: 'yes'}
  CALC: {0: 'Always', 1: 'Frequently', 2: 'Sometimes', 3: 'no'}
  MTRANS: {0: 'Automobile', 1: 'Bike', 2: 'Motorbike', 3: 'Public_Transportation', 4: 'Walking'}
  NObeyesdad: {0: 'Insufficient_Weight', 1: 'Normal_Weight', 2: 'Obesity_Type_I', 3: 'Obesity_Type_II', 4: 'Obesity_Type_III', 5: 'Overweight_Level_I', 6: 'Overweight_Level_II'}

Saved: models/label_encoders.pkl


In [5]:
# ── 4. StandardScaler (hanya fitur, bukan target) ─────────────────
TARGET   = 'NObeyesdad'
FEATURES = [c for c in obesity_enc.columns if c != TARGET]

scaler = StandardScaler()
obesity_enc[FEATURES] = scaler.fit_transform(obesity_enc[FEATURES])

with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Saved: models/scaler.pkl')

# Simpan feature list — penting untuk urutan kolom saat inference
model_meta = {
    'features': FEATURES,
    'target': TARGET,
    'classes': le_dict[TARGET].classes_.tolist()
}
with open('../models/model_meta.json', 'w') as f:
    json.dump(model_meta, f, indent=2)
print('Saved: models/model_meta.json')
print('\nClasses:', model_meta['classes'])

Saved: models/scaler.pkl
Saved: models/model_meta.json

Classes: ['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I', 'Overweight_Level_II']


In [6]:
# ── 5. Save ML-ready version ──────────────────────────────────────
obesity_enc.to_csv('../data/processed/obesity_encoded.csv', index=False)
print('Saved: obesity_encoded.csv  (encoded + scaled, siap untuk training)')
print(f'Final shape: {obesity_enc.shape}')
obesity_enc.head(3)

Saved: obesity_encoded.csv  (encoded + scaled, siap untuk training)
Final shape: (2087, 21)


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,...,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad,BMI,BMR,ActivityLevel,DailyCalorieTarget
0,-1.008179,-0.526613,-0.887408,-0.872985,0.460394,-2.754719,-0.788364,0.390906,0.318128,-0.146755,...,-0.219584,-1.186977,0.554211,1.411836,0.506768,1,-0.670040,-1.045942,-1.123103,-1.328131
1,-1.008179,-0.526613,-1.960788,-1.178508,0.460394,-2.754719,1.082164,0.390906,0.318128,6.814090,...,4.554073,2.328908,-1.090505,-0.522834,0.506768,1,-0.688736,-1.345764,2.215808,-0.318153
2,0.991887,-0.212507,1.044677,-0.376509,0.460394,-2.754719,-0.788364,0.390906,0.318128,-0.146755,...,-0.219584,1.156947,0.554211,-2.457503,0.506768,1,-0.747317,0.104756,1.102838,0.639675


---
## BAGIAN 2 — Food & Nutrition Dataset
### Tujuan: Bersihkan sebagai Recommendation Database

**Kolom dari EDA (12 kolom, 645 rows, 0 missing):**
`Food_Item, Category, Calories (kcal), Protein (g), Carbohydrates (g),`  
`Fat (g), Fiber (g), Sugars (g), Sodium (mg), Cholesterol (mg), Meal_Type, Water_Intake (ml)`

**Langkah:**
1. Rename kolom → snake_case
2. Filter baris dengan kalori = 0
3. Tambah kolom `calorie_category` untuk filtering UI
4. Tambah kolom `health_score` sebagai ranking helper

In [7]:
# ── Load ──────────────────────────────────────────────────────────
food = pd.read_csv('..\\data\\raw\\daily_food_nutrition_dataset.csv', on_bad_lines='skip')
print(f'Original shape: {food.shape}')  # (645, 12)
print('No missing values confirmed from EDA.')
food.head(3)

Original shape: (645, 12)
No missing values confirmed from EDA.


,Food_Item,Category,Calories (kcal),Protein (g),Carbohydrates (g),Fat (g),Fiber (g),Sugars (g),Sodium (mg),Cholesterol (mg),Meal_Type,Water_Intake (ml)
0,Scrambled Eggs (2 large),Protein/Dairy,180,12.0,2.0,14.0,0.0,1.0,180,370,Breakfast,250
1,Whole Wheat Toast (1 slice),Grain,80,4.0,14.0,1.0,2.0,2.0,140,0,Breakfast,0
2,Coffee (black),Beverage,5,0.3,0.0,0.1,0.0,0.0,5,0,Breakfast,0


In [8]:
# ── 1. Rename kolom → snake_case uniform ─────────────────────────
rename_map = {
    'Food_Item':           'food_name',
    'Category':            'category',
    'Calories (kcal)':     'calories',
    'Protein (g)':         'protein_g',
    'Carbohydrates (g)':   'carbs_g',
    'Fat (g)':             'fat_g',
    'Fiber (g)':           'fiber_g',
    'Sugars (g)':          'sugars_g',
    'Sodium (mg)':         'sodium_mg',
    'Cholesterol (mg)':    'cholesterol_mg',
    'Meal_Type':           'meal_type',
    'Water_Intake (ml)':   'water_ml'
}
food.rename(columns=rename_map, inplace=True)
print('Renamed columns:', food.columns.tolist())

Renamed columns: ['food_name', 'category', 'calories', 'protein_g', 'carbs_g', 'fat_g', 'fiber_g', 'sugars_g', 'sodium_mg', 'cholesterol_mg', 'meal_type', 'water_ml']


In [9]:
# ── 2. Filter baris anomali ───────────────────────────────────────
# Dari EDA: min calories = 0.0, min protein = 0.0, dll
# Baris dengan calories=0 tidak berguna sebagai rekomendasi makanan
before = len(food)
food = food[food['calories'] > 0].copy()
print(f'Removed {before - len(food)} rows with calories=0. Remaining: {len(food)}')

# Drop duplikat food_name
food.drop_duplicates(subset='food_name', inplace=True)
food.reset_index(drop=True, inplace=True)
print(f'After dedup: {food.shape}')

Removed 6 rows with calories=0. Remaining: 639
After dedup: (581, 12)


In [10]:
# ── 3. Tambah calorie_category untuk filtering di UI ─────────────
# Ini digunakan saat filter makanan berdasarkan target kalori harian user
def calorie_category(kcal):
    if kcal < 100:
        return 'low'      # camilan ringan
    elif kcal < 300:
        return 'medium'   # makanan ringan / sarapan
    elif kcal < 500:
        return 'high'     # makan besar
    else:
        return 'very_high'

food['calorie_category'] = food['calories'].apply(calorie_category)
print('Calorie category distribution:')
print(food['calorie_category'].value_counts())

Calorie category distribution:
calorie_category
low          269
medium       230
high          58
very_high     24
Name: count, dtype: int64


In [11]:
# ── 4. Health Score — ranking helper untuk rekomendasi ────────────
# Formula: reward protein & fiber, penalize sugar, sodium, cholesterol
# Dinormalisasi ke 0-100

def normalize(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

food['health_score'] = (
    normalize(food['protein_g'])   * 30 +
    normalize(food['fiber_g'])     * 25 -
    normalize(food['sugars_g'])    * 20 -
    normalize(food['sodium_mg'])   * 15 -
    normalize(food['cholesterol_mg']) * 10
).round(2)

# Rescale ke 0-100
hs = food['health_score']
food['health_score'] = ((hs - hs.min()) / (hs.max() - hs.min()) * 100).round(1)

print('Health score stats:')
print(food['health_score'].describe())
print('\nTop 5 healthiest foods:')
print(food.nlargest(5, 'health_score')[['food_name','calories','protein_g','fiber_g','health_score']])

Health score stats:
count    581.000000
mean      48.792771
std       15.484179
min        0.000000
25%       40.500000
50%       44.600000
75%       55.300000
max      100.000000
Name: health_score, dtype: float64

Top 5 healthiest foods:
                     food_name  calories  protein_g  fiber_g  health_score
27    Beef Steak (6oz sirloin)       270       42.0      0.0         100.0
19         Lentil Soup (1 can)       230       15.0     12.0          96.9
484      Osso Buco (1 serving)       550       45.0      4.0          96.9
473  Ribollita (1.5 cups soup)       350       15.0     14.0          96.3
208        Hummus Wrap (large)       450       15.0     12.0          94.3


In [12]:
# ── 5. Cek & standarisasi meal_type dan category ──────────────────
print('Meal types:', food['meal_type'].unique())
print('Categories:', food['category'].unique())

food['meal_type'] = food['meal_type'].str.strip().str.lower()
food['category']  = food['category'].str.strip().str.lower()

# Save
food.to_csv('../data/processed/food_clean.csv', index=False)
print(f'\nSaved: food_clean.csv  shape={food.shape}')
food.head(3)

Meal types: ['Breakfast' 'Lunch' 'Snack' 'Dinner' 'Side']
Categories: ['Protein/Dairy' 'Grain' 'Beverage' 'Fruit' 'Meal/Protein' 'Protein/Fish'
 'Vegetable' 'Dairy' 'Nut' 'Meal/Processed' 'Meal/Pasta' 'Supplement'
 'Grain/Processed' 'Meal/Legume' 'Meal/Vegetarian' 'Meal/Fish'
 'Protein/Meat' 'Dairy/Dessert' 'Protein' 'Vegetable/Processed'
 'Supplement/Processed' 'Legume' 'Dessert' 'Condiment' 'Beverage/Meal'
 'Protein/Vegetarian' 'Meal/Vegetable' 'Meal/Meat' 'Vegetable/Condiment'
 'Meal/Seafood' 'Beverage/Dairy' 'Grain/Dessert' 'Snack/Processed'
 'Condiment/Dairy' 'Protein/Seafood' 'Meal/Grain' 'Meal'
 'Beverage/Dairy-Alt' 'Protein/Processed' 'Snack' 'Condiment/Processed'
 'Snack/Dessert' 'Meal/Fruit' 'Meal/Soup' 'Meal/Rice' 'Snack/Appetizer']

Saved: food_clean.csv  shape=(581, 14)


,food_name,category,calories,protein_g,carbs_g,fat_g,fiber_g,sugars_g,sodium_mg,cholesterol_mg,meal_type,water_ml,calorie_category,health_score
0,Scrambled Eggs (2 large),protein/dairy,180,12.0,2.0,14.0,0.0,1.0,180,370,breakfast,250,medium,37.0
1,Whole Wheat Toast (1 slice),grain,80,4.0,14.0,1.0,2.0,2.0,140,0,breakfast,0,low,52.3
2,Coffee (black),beverage,5,0.3,0.0,0.1,0.0,0.0,5,0,breakfast,0,low,42.7


---
## BAGIAN 3 — Gym Exercise Dataset
### Tujuan: Bersihkan sebagai Recommendation Database

**Kolom dari EDA (9 kolom, 2918 rows):**  
`Unnamed:0, Title, Desc, Type, BodyPart, Equipment, Level, Rating, RatingDesc`

**Missing values dari EDA:**
- `Desc`: 1550 missing
- `Equipment`: 32 missing  
- `Rating`: 1887 missing
- `RatingDesc`: 2056 missing

**Langkah:**
1. Drop kolom tidak berguna (`Unnamed:0`, `RatingDesc`)
2. Handle missing values sesuai konteks kolom
3. Normalize `Level`, `Type`, `BodyPart` → lowercase
4. Buat mapping `obesity_class → recommended_level`
5. Save

In [13]:
# ── Load ──────────────────────────────────────────────────────────
gym = pd.read_csv('..\\data\\raw\\megaGymDataset.csv')
print(f'Original shape: {gym.shape}')  # (2918, 9)
gym.head(3)

Original shape: (2918, 9)


,Unnamed: 0,Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


In [14]:
# ── 1. Drop kolom tidak berguna ───────────────────────────────────
# 'Unnamed: 0' = index duplikat
# 'RatingDesc' = 2056/2918 missing, hanya nilai 'Average' → tidak informatif
gym.drop(columns=['Unnamed: 0', 'RatingDesc'], inplace=True)
print('Remaining columns:', gym.columns.tolist())

Remaining columns: ['Title', 'Desc', 'Type', 'BodyPart', 'Equipment', 'Level', 'Rating']


In [15]:
# ── 2. Handle missing values ──────────────────────────────────────

# Title: 0 missing, ini primary key → drop jika ada null
gym.dropna(subset=['Title'], inplace=True)

# Desc: 1550 missing → isi dengan string kosong
# (tidak kritis, hanya untuk tampilan UI)
gym['Desc'] = gym['Desc'].fillna('')

# Equipment: 32 missing → isi 'Body Only' (bodyweight exercise)
# Logis karena exercise tanpa equipment sering cukup dengan bodyweight
gym['Equipment'] = gym['Equipment'].fillna('Body Only')

# Rating: 1887/2918 missing (~65%) → terlalu banyak untuk diisi
# Strategi: isi dengan median rating dari data yang ada
median_rating = gym['Rating'].median()
gym['Rating'] = gym['Rating'].fillna(median_rating).round(1)
print(f'Rating missing filled with median: {median_rating}')

# Drop duplikat Title
before = len(gym)
gym.drop_duplicates(subset='Title', inplace=True)
gym.reset_index(drop=True, inplace=True)
print(f'Removed {before - len(gym)} duplicate titles. Remaining: {len(gym)}')

print('\nMissing after handling:')
print(gym.isnull().sum())

Rating missing filled with median: 7.9
Removed 9 duplicate titles. Remaining: 2909

Missing after handling:
Title        0
Desc         0
Type         0
BodyPart     0
Equipment    0
Level        0
Rating       0
dtype: int64


In [16]:
# ── 3. Normalize string kolom → lowercase strip ───────────────────
str_cols = ['Type', 'BodyPart', 'Equipment', 'Level']
for col in str_cols:
    gym[col] = gym[col].str.strip().str.lower()

print('Level values:   ', sorted(gym['Level'].unique()))
print('Type values:    ', sorted(gym['Type'].unique()))
print('BodyPart unique:', gym['BodyPart'].nunique(), '->', sorted(gym['BodyPart'].unique()))

Level values:    ['beginner', 'expert', 'intermediate']
Type values:     ['cardio', 'olympic weightlifting', 'plyometrics', 'powerlifting', 'strength', 'stretching', 'strongman']
BodyPart unique: 17 -> ['abdominals', 'abductors', 'adductors', 'biceps', 'calves', 'chest', 'forearms', 'glutes', 'hamstrings', 'lats', 'lower back', 'middle back', 'neck', 'quadriceps', 'shoulders', 'traps', 'triceps']


In [17]:
# ── 4. Mapping: Obesity Class → Recommended Exercise Level ────────
# Ini adalah JEMBATAN antara hasil prediksi ML dan gym database
# Logic:
#  - Berat badan sangat berlebih (Obesity II/III) → beginner dulu
#  - Normal weight → bisa intermediate
#  - Underweight → beginner (fokus strength)

obesity_to_exercise = {
    'Insufficient_Weight': {
        'level': 'beginner',
        'preferred_type': ['strength'],
        'note': 'Fokus pada latihan kekuatan untuk menambah massa otot'
    },
    'Normal_Weight': {
        'level': 'intermediate',
        'preferred_type': ['strength', 'cardio'],
        'note': 'Kombinasi cardio dan strength untuk menjaga komposisi tubuh'
    },
    'Overweight_Level_I': {
        'level': 'beginner',
        'preferred_type': ['cardio', 'stretching'],
        'note': 'Mulai dengan cardio ringan dan stretching'
    },
    'Overweight_Level_II': {
        'level': 'beginner',
        'preferred_type': ['cardio', 'stretching'],
        'note': 'Prioritaskan cardio intensitas rendah'
    },
    'Obesity_Type_I': {
        'level': 'beginner',
        'preferred_type': ['cardio', 'stretching'],
        'note': 'Cardio low-impact seperti jalan kaki atau berenang'
    },
    'Obesity_Type_II': {
        'level': 'beginner',
        'preferred_type': ['stretching', 'cardio'],
        'note': 'Konsultasikan dengan profesional, mulai dari stretching'
    },
    'Obesity_Type_III': {
        'level': 'beginner',
        'preferred_type': ['stretching'],
        'note': 'Sangat disarankan konsultasi dokter sebelum olahraga'
    }
}

with open('../data/processed/obesity_to_exercise_map.json', 'w') as f:
    json.dump(obesity_to_exercise, f, indent=2, ensure_ascii=False)
print('Saved: obesity_to_exercise_map.json')
print(json.dumps(obesity_to_exercise, indent=2, ensure_ascii=False))

Saved: obesity_to_exercise_map.json
{
  "Insufficient_Weight": {
    "level": "beginner",
    "preferred_type": [
      "strength"
    ],
    "note": "Fokus pada latihan kekuatan untuk menambah massa otot"
  },
  "Normal_Weight": {
    "level": "intermediate",
    "preferred_type": [
      "strength",
      "cardio"
    ],
    "note": "Kombinasi cardio dan strength untuk menjaga komposisi tubuh"
  },
  "Overweight_Level_I": {
    "level": "beginner",
    "preferred_type": [
      "cardio",
      "stretching"
    ],
    "note": "Mulai dengan cardio ringan dan stretching"
  },
  "Overweight_Level_II": {
    "level": "beginner",
    "preferred_type": [
      "cardio",
      "stretching"
    ],
    "note": "Prioritaskan cardio intensitas rendah"
  },
  "Obesity_Type_I": {
    "level": "beginner",
    "preferred_type": [
      "cardio",
      "stretching"
    ],
    "note": "Cardio low-impact seperti jalan kaki atau berenang"
  },
  "Obesity_Type_II": {
    "level": "beginner",
    "preferr

In [18]:
# ── 5. Verifikasi: ada data untuk setiap level? ───────────────────
print('Exercise counts per level:')
print(gym['Level'].value_counts())
print('\nExercise counts per type:')
print(gym['Type'].value_counts())

# Contoh: ambil 3 rekomendasi untuk Obesity_Type_I
sample_map  = obesity_to_exercise['Obesity_Type_I']
sample_recs = gym[
    (gym['Level'] == sample_map['level']) &
    (gym['Type'].isin(sample_map['preferred_type']))
].sample(min(3, len(gym)), random_state=42)[['Title','Type','BodyPart','Level']]

print('\nSample rekomendasi untuk Obesity_Type_I:')
print(sample_recs.to_string(index=False))

Exercise counts per level:
Level
intermediate    2437
beginner         459
expert            13
Name: count, dtype: int64

Exercise counts per type:
Type
strength                 2536
stretching                147
plyometrics                97
powerlifting               37
cardio                     35
olympic weightlifting      35
strongman                  22
Name: count, dtype: int64

Sample rekomendasi untuk Obesity_Type_I:
                               Title       Type   BodyPart    Level
                            Groiners stretching  adductors beginner
                    Football Up-Down     cardio quadriceps beginner
Standing Soleus And Achilles Stretch stretching     calves beginner


In [19]:
# ── Save ──────────────────────────────────────────────────────────
gym.to_csv('../data/processed/gym_clean.csv', index=False)
print(f'Saved: gym_clean.csv  shape={gym.shape}')
gym.head(3)

Saved: gym_clean.csv  shape=(2909, 7)


,Title,Desc,Type,BodyPart,Equipment,Level,Rating
0,Partner plank band row,The partner plank band row is an abdominal exe...,strength,abdominals,bands,intermediate,0.0
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,strength,abdominals,bands,intermediate,7.9
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,strength,abdominals,bands,intermediate,7.9
